# Strategy A — One-Shot Code Generation

Models: Qwen2.5-Coder-7B-Instruct and DeepSeek-Coder-V2-Lite-Instruct.

Dataset: 10 MBPP programming tasks.

Each model generates one solution per task without feedback or correction.

**Important:** This notebook contains the original generation implementation extracted from the experiment notebook. It has not been rerun to produce the saved results.

The original experiment used slightly different prompt formatting for Qwen and DeepSeek.

Do not rerun the generation cells unless performing a separate, explicitly documented experiment.


In [ ]:
import json
from pathlib import Path
from collections import Counter

dataset_path = Path(
    "/home/mahkot/Team-Horizon/Dataset/mbpp_initial_10.jsonl"
)

tasks = []

with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            tasks.append(json.loads(line))

print("=== INITIAL DATASET ===")
print("Total tasks:", len(tasks))

print("\n=== TASK DETAILS ===")

for task in tasks:
    task_id = task.get("task_id")
    difficulty = task.get("difficulty", "Not specified")
    tests = task.get("test_list", [])

    print(
        f"Task ID: {task_id} | "
        f"Difficulty: {difficulty} | "
        f"Tests: {len(tests)}"
    )

print("\n=== VALIDATION ===")

task_ids = [task.get("task_id") for task in tasks]

print("Unique task IDs:", len(set(task_ids)))
print("Total test cases:", sum(
    len(task.get("test_list", [])) for task in tasks
))

print("Difficulty distribution:", dict(Counter(
    task.get("difficulty", "Not specified") for task in tasks
)))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Loading Qwen tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

print("Loading Qwen model...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    local_files_only=True
)

model.eval()

print("\n=== MODEL LOADED SUCCESSFULLY ===")
print("Model:", model.config._name_or_path)
print("Device:", next(model.parameters()).device)
print("Data type:", next(model.parameters()).dtype)
print("Ready for inference:", not model.training)

In [ ]:
import ast
import re

def clean_generated_code(raw_code):
    code = raw_code.strip()

    # Remove opening Markdown code fence
    code = re.sub(r"^```(?:python|py)?\s*\n?", "", code)

    # Remove closing Markdown code fence if present
    code = re.sub(r"\n?```\s*$", "", code)

    return code.strip()


cleaned_code = clean_generated_code(generated_code)

print("=== CLEANED CODE ===")
print(cleaned_code)

print("\n=== SYNTAX VALIDATION ===")

try:
    ast.parse(cleaned_code)
    print("Python syntax: VALID")
except SyntaxError as e:
    print("Python syntax: INVALID")
    print("Error:", e)

In [ ]:
import json
import ast
import time
import torch
from pathlib import Path

results_dir = Path("/home/mahkot/Team-Horizon/Results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "qwen_initial_10_generations.jsonl"

# Prevent accidental overwriting of completed experiments
if output_file.exists():
    raise FileExistsError(
        f"{output_file} already exists. Inspect it before rerunning."
    )

generation_results = []

model.eval()

for index, task in enumerate(tasks, start=1):

    print(f"\nGenerating task {index}/10 — ID {task['task_id']}")

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Python programmer. "
                "Write a correct Python function for the given task. "
                "Return only Python code without explanations or Markdown."
            )
        },
        {
            "role": "user",
            "content": task["prompt"]
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to("cuda:0")

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    raw_code = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    code = clean_generated_code(raw_code)

    try:
        ast.parse(code)
        syntax_valid = True
        syntax_error = None
    except SyntaxError as exc:
        syntax_valid = False
        syntax_error = str(exc)

    result = {
        "model": model_name,
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "prompt": task["prompt"],
        "raw_output": raw_code,
        "cleaned_code": code,
        "generation_time_seconds": round(elapsed, 4),
        "output_tokens": len(new_tokens),
        "syntax_valid": syntax_valid,
        "syntax_error": syntax_error,
        "tests_passed": None,
        "tests_total": len(task["test_list"]),
        "evaluation_status": "not_executed",
        "generation_settings": {
            "max_new_tokens": 256,
            "do_sample": False
        }
    }

    generation_results.append(result)

    print(
        f"  Time: {elapsed:.2f}s | "
        f"Tokens: {len(new_tokens)} | "
        f"Syntax: {'VALID' if syntax_valid else 'INVALID'}"
    )

# Save all results only after generation completes
with open(output_file, "w", encoding="utf-8") as f:
    for result in generation_results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print("\n=== QWEN GENERATION COMPLETED ===")
print("Total tasks:", len(generation_results))
print("Valid syntax:", sum(r["syntax_valid"] for r in generation_results))
print("Saved to:", output_file)

In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/tmp/team_horizon_deepseek_model"

print("=== LOADING DEEPSEEK ===")

start_time = time.perf_counter()

deepseek_tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
    local_files_only=True
)

deepseek_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    local_files_only=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map={"": "cuda:2"},
    low_cpu_mem_usage=True
)

deepseek_model.eval()

load_time = time.perf_counter() - start_time

print("\n=== DEEPSEEK LOADED ===")
print(f"Loading time: {load_time:.2f} seconds")
print("Model device:", next(deepseek_model.parameters()).device)

print(
    "GPU 2 allocated memory:",
    f"{torch.cuda.memory_allocated(2) / (1024**3):.2f} GB"
)

In [ ]:
import json
from pathlib import Path

dataset_path = Path(
    "/home/mahkot/Team-Horizon/Dataset/mbpp_initial_10.jsonl"
)

with dataset_path.open("r", encoding="utf-8") as f:
    deepseek_tasks = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("=== DEEPSEEK DATASET VERIFICATION ===")
print("Total tasks:", len(deepseek_tasks))

for task in deepseek_tasks:
    print(
        "Task ID:", task["task_id"],
        "| Difficulty:", task["difficulty"],
        "| Tests:", len(task["test_list"])
    )

print("\n=== RESULTS DIRECTORY ===")
results_dir = Path("/home/mahkot/Team-Horizon/Results")
print("Exists:", results_dir.is_dir())

In [ ]:
import ast
import json
import re
import time
import torch
from pathlib import Path

output_path = Path(
    "/home/mahkot/Team-Horizon/Results/"
    "deepseek_initial_10_generations.jsonl"
)

# Prevent accidental overwriting
if output_path.exists():
    raise FileExistsError(
        f"Results already exist: {output_path}"
    )

def required_function_name(task):
    for test in task["test_list"]:
        tree = ast.parse(test)
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name) and node.func.id == "assert":
                    continue
                if isinstance(node.func, ast.Name):
                    return node.func.id
    raise ValueError(f"No function found for task {task['task_id']}")

def clean_code(text):
    text = text.strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )
    return match.group(1).strip() if match else text

deepseek_results = []

print("=== DEEPSEEK 10-TASK GENERATION ===")

with output_path.open("x", encoding="utf-8") as output_file:

    for task in deepseek_tasks:

        function_name = required_function_name(task)

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert Python programmer. "
                    "Write a correct Python function for the given task. "
                    "Use exactly the required function name. "
                    "Return only Python code without explanations or Markdown."
                )
            },
            {
                "role": "user",
                "content": (
                    f"{task['prompt']}\n\n"
                    f"Required function name: {function_name}"
                )
            }
        ]

        prompt = deepseek_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = deepseek_tokenizer(
            prompt,
            return_tensors="pt"
        ).to("cuda:2")

        torch.cuda.synchronize(2)
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = deepseek_model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                use_cache=False,
                pad_token_id=deepseek_tokenizer.eos_token_id
            )

        torch.cuda.synchronize(2)
        elapsed = time.perf_counter() - start

        tokens = outputs[0][inputs["input_ids"].shape[1]:]

        raw_code = deepseek_tokenizer.decode(
            tokens,
            skip_special_tokens=True
        )

        cleaned_code = clean_code(raw_code)

        syntax_valid = True
        syntax_error = None
        function_match = False

        try:
            parsed = ast.parse(cleaned_code)
            function_match = any(
                isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
                and node.name == function_name
                for node in parsed.body
            )
        except SyntaxError as e:
            syntax_valid = False
            syntax_error = str(e)

        result = {
            "model": "DeepSeek-Coder-V2-Lite-Instruct",
            "task_id": task["task_id"],
            "difficulty": task["difficulty"],
            "prompt": task["prompt"],
            "required_function_name": function_name,
            "raw_output": raw_code,
            "cleaned_code": cleaned_code,
            "generation_time_seconds": round(elapsed, 4),
            "output_tokens": len(tokens),
            "syntax_valid": syntax_valid,
            "syntax_error": syntax_error,
            "function_name_match": function_match,
            "tests_passed": None,
            "tests_total": len(task["test_list"]),
            "evaluation_status": "not_executed",
            "generation_settings": {
                "max_new_tokens": 256,
                "do_sample": False,
                "use_cache": False,
                "dtype": "bfloat16",
                "attention": "eager"
            }
        }

        deepseek_results.append(result)

        output_file.write(json.dumps(result) + "\n")
        output_file.flush()

        print(
            f"Task {task['task_id']} "
            f"({task['difficulty']}): "
            f"syntax={syntax_valid}, "
            f"name_match={function_match}, "
            f"time={elapsed:.2f}s, "
            f"tokens={len(tokens)}",
            flush=True
        )

print("\n=== GENERATION COMPLETE ===")
print("Total generated:", len(deepseek_results))
print("Syntax valid:", sum(r["syntax_valid"] for r in deepseek_results))
print("Function names matched:", sum(r["function_name_match"] for r in deepseek_results))
print("Saved to:", output_path)